In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.miller.lesson3 import Exercise

In [ ]:
def normalize_data(X_train, X_val, X_test):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    eps = 1e-8
    std = np.where(std < eps, 1.0, std)

    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std

    return X_train_norm, X_val_norm, X_test_norm, mean, std


def create_digits_model(input_size: int = 64, hidden_sizes: list[int] | None = None, output_size: int = 10):
    if hidden_sizes is None:
        hidden_sizes = [128, 64]

    layers = []
    layers.append(Exercise.create_linear_layer(input_size, hidden_sizes[0]))
    layers.append(Exercise.create_relu_layer())

    for i in range(len(hidden_sizes) - 1):
        layers.append(Exercise.create_linear_layer(hidden_sizes[i], hidden_sizes[i + 1]))
        layers.append(Exercise.create_relu_layer())

    layers.append(Exercise.create_linear_layer(hidden_sizes[-1], output_size))
    layers.append(Exercise.create_logsoftmax_layer())

    return Exercise.create_model(*layers)


def evaluate_model(model, X, y):
    log_probs = model.forward(X)  # выход LogSoftmax
    loss = -np.mean(log_probs[np.arange(len(y)), y])  # NLL
    predictions = np.argmax(log_probs, axis=1)
    accuracy = np.mean(predictions == y)
    return loss, accuracy


def main():
    digits = load_digits()
    X = digits.data.astype(np.float32)
    y = digits.target

    print(f"Данные: {X.shape}")

    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    print(f"Train: {X_train.shape[0]}")
    print(f"Val:   {X_val.shape[0]}")
    print(f"Test:  {X_test.shape[0]}")

    X_train_norm, X_val_norm, X_test_norm, mean, std = normalize_data(X_train, X_val, X_test)

    model = create_digits_model(input_size=64, hidden_sizes=[128, 64], output_size=10)
    loss_fn = Exercise.create_nll_loss()

    Exercise.train_model(model=model, loss=loss_fn, x=X_train_norm, y=y_train, lr=0.01, n_epoch=100, batch_size=32)

    test_loss, test_acc = evaluate_model(model, X_test_norm, y_test)
    print(f"\nТестовые потери: {test_loss:.4f}")
    print(f"Тестовая точность: {test_acc:.4f}")

    log_probs = model.forward(X_test_norm)
    predictions = np.argmax(log_probs, axis=1)

    for digit in range(10):
        mask = y_test == digit
        if np.sum(mask) > 0:
            acc = np.mean(predictions[mask] == digit)
            print(f"{digit}: {acc:.4f} ({np.sum(mask)} примеров)")

    return model, test_acc


if __name__ == "__main__":
    model, accuracy = main()